In [ ]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
!pip3 install lerobot huggingface_hub lerobot[libero] h5py zarr transformers numpy tqdm diffusers timm matplotlib

Looking in indexes: https://download.pytorch.org/whl/cu130


In [ ]:
!git clone https://github.com/GoncaloMark/DuPLO-VLA.git

Cloning into 'DuPLO-VLA'...
remote: Enumerating objects: 1532, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 1532 (delta 114), reused 135 (delta 55), pack-reused 1285 (from 1)
Receiving objects: 100% (1532/1532), 166.08 MiB | 18.93 MiB/s, done.
Resolving deltas: 100% (702/702), done.


In [ ]:
%cd DuPLO-VLA

/content/DuPLO-VLA


In [ ]:
!git switch v2
!git pull

Branch 'v2' set up to track remote branch 'v2' from 'origin'.
Switched to a new branch 'v2'
Already up to date.


In [ ]:
import sys, os
sys.path.append('/content/DuPLO-VLA/src')
os.chdir('/content/DuPLO-VLA')

import numpy as np
import torch
import h5py
from pathlib import Path
from tqdm.notebook import tqdm

from vlm.vlm import VisualTaskPlanner

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi, login
from google.colab import drive, runtime


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
login()

In [ ]:
out_dataset_id   = "lnxdre4d/libero_features_v2"
target_suite     = "libero_goal"
in_dataset_repo  = "lerobot/libero"

vlm_stride       = 8
batch_size       = 256
layer_indices    = [4, 12, 25, 34]
vlm_dim          = 2560

out_h5_path = "/content/libero_features_v2.h5"

In [ ]:
dataset = LeRobotDataset(in_dataset_repo)

print(f"Episodes: {dataset.num_episodes}")
print(f"Frames: {dataset.num_frames}")
print(f"Tasks: {len(dataset.meta.tasks)}")
print()
print("Sample of task names:")
for idx, task in list(dataset.meta.tasks.items())[:20]:
    print(f"  [{idx}] {task}")
print()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 457 files:   0%|          | 0/457 [00:00<?, ?it/s]

Episodes: 1693
Frames: 273465
Tasks: 40

Sample of task names:
  [task_index] task
put the white mug on the left plate and put the yellow and white mug on the right plate      0
put the white mug on the plate and put the chocolate pudding to the right of the plate       1
put the yellow and white mug in the microwave and close it                                   2
turn on the stove and put the moka pot on it                                                 3
put both the alphabet soup and the cream cheese box in the basket                            4
put both the alphabet soup and the tomato sauce in the basket                                5
put both moka pots on the stove                                                              6
put both the cream cheese box and the butter in the basket                                   7
put the black bowl in the bottom drawer of the cabinet and close it                          8
pick up the book and place it in the back compartment of the c

In [ ]:
sample = dataset[0]
print("Sample keys:")
for k in sample.keys():
    v = sample[k]
    if hasattr(v, "shape"):
        print(f"  {k}: shape={tuple(v.shape)} dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v).__name__}")

Sample keys:
  observation.images.image: shape=(3, 256, 256) dtype=torch.float32
  observation.images.image2: shape=(3, 256, 256) dtype=torch.float32
  observation.state: shape=(8,) dtype=torch.float32
  action: shape=(7,) dtype=torch.float32
  timestamp: shape=() dtype=torch.float32
  frame_index: shape=() dtype=torch.int64
  episode_index: shape=() dtype=torch.int64
  index: shape=() dtype=torch.int64
  task_index: shape=() dtype=torch.int64
  task: str


In [ ]:
candidates_main  = ["observation.images.image", "observation.images.agentview_image"]
candidates_wrist = ["observation.images.image_wrist", "observation.images.image2", "observation.images.wrist_image",
                    "observation.images.eye_in_hand_image"]

main_camera_key  = next((k for k in candidates_main  if k in sample), None)
wrist_camera_key = next((k for k in candidates_wrist if k in sample), None)

if main_camera_key is None:
    raise RuntimeError(f"No main-camera key found. Candidates: {candidates_main}. "
                       f"Available: {list(sample.keys())}")
if wrist_camera_key is None:
    raise RuntimeError(f"No wrist-camera key found. Candidates: {candidates_wrist}. "
                       f"Available: {list(sample.keys())}")

print(f"Using main camera : {main_camera_key}")
print(f"Using wrist camera: {wrist_camera_key}")


Using main camera : observation.images.image
Using wrist camera: observation.images.image2


In [ ]:
planner = VisualTaskPlanner(load_vlm=True, num_pooling_queries=64)
planner.eval().to("cuda")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

VisualTaskPlanner(
  (vlm): Qwen3VLForConditionalGeneration(
    (model): Qwen3VLModel(
      (visual): Qwen3VLVisionModel(
        (patch_embed): Qwen3VLVisionPatchEmbed(
          (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
        )
        (pos_embed): Embedding(2304, 1024)
        (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
        (blocks): ModuleList(
          (0-23): 24 x Qwen3VLVisionBlock(
            (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (attn): Qwen3VLVisionAttention(
              (qkv): Linear(in_features=1024, out_features=3072, bias=True)
              (proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (mlp): Qwen3VLVisionMLP(
              (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
              (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)


In [ ]:
def compute_vlm_indices(T: int, stride: int):
    """Slow-frame source indices and per-env-frame slow lookup."""
    slow_source_idx = np.arange(0, T, stride, dtype=np.int64)
    per_frame_slow  = np.empty(T, dtype=np.int64)
    for slow_i, src in enumerate(slow_source_idx):
        nxt = slow_source_idx[slow_i + 1] if slow_i + 1 < len(slow_source_idx) else T
        per_frame_slow[src:nxt] = slow_i
    return slow_source_idx, per_frame_slow


def bf16_to_uint16(t: torch.Tensor) -> np.ndarray:
    assert t.dtype == torch.bfloat16
    return t.contiguous().view(torch.uint16).cpu().numpy()


def to_hwc_uint8_float(img):
    """LeRobot images can be CHW or HWC, uint8 or float [0,1]. Normalize to
    HWC float32 in [0, 255]."""
    if isinstance(img, torch.Tensor):
        img = img.cpu().numpy()
    if img.ndim != 3:
        raise ValueError(f"Expected 3D image, got {img.shape}")
    if img.shape[0] == 3 and img.shape[-1] != 3:
        img = np.transpose(img, (1, 2, 0))
    if img.dtype != np.uint8:
        img = np.clip(img * 255.0, 0, 255).astype(np.float32)
    else:
        img = img.astype(np.float32)
    return img


@torch.no_grad()
def extract_slow_stream(planner, imgs_main_slow, imgs_wrist_slow, instruction,
                        batch_size: int, layer_indices):
    """Run the VLM on slow-frame pairs (main, wrist) and return:
       hidden: (T_slow, num_layers, L_total, vlm_dim) bf16
       mask: (T_slow, L_total) bool
    """
    out_hidden = []
    out_mask   = []

    n = len(imgs_main_slow)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        main_chunk  = list(imgs_main_slow[start:end])
        wrist_chunk = list(imgs_wrist_slow[start:end])
        texts       = [instruction] * (end - start)

        all_h, mask = planner.extract_features_batch(main_chunk, wrist_chunk, texts)
        sampled = torch.stack([all_h[i] for i in layer_indices], dim=1)

        out_hidden.append(sampled)
        out_mask.append(mask.bool())

    hidden = torch.cat(out_hidden, dim=0)
    mask   = torch.cat(out_mask,   dim=0)
    return hidden, mask


In [ ]:
ep_to_task = {}
for ep_idx in range(dataset.num_episodes):
    ep_meta = dataset.meta.episodes[ep_idx]

    if isinstance(ep_meta, dict) and "tasks" in ep_meta:
        name = ep_meta["tasks"][0]
        ti = 0
    elif isinstance(ep_meta, dict) and "task_index" in ep_meta:
        ti = ep_meta["task_index"]
        name = dataset.meta.tasks[ti]
    else:
        from_idx = (ep_meta.get("dataset_from_index", ep_meta.get("from_index", 0))
                    if isinstance(ep_meta, dict)
                    else getattr(ep_meta, "dataset_from_index",
                                 getattr(ep_meta, "from_index", 0)))
        first = dataset[from_idx]
        if "task" in first and isinstance(first["task"], str):
            name = first["task"]
            ti = first.get("task_index", 0)
            if hasattr(ti, "item"):
                ti = ti.item()
        else:
            ti, name = 0, "unknown_task"

    ep_to_task[ep_idx] = (int(ti), name)

print(f"Built ep_to_task with {len(ep_to_task)} entries")

SUITE_TASK_RANGES = {
    "libero_10":      range(0, 10),
    "libero_goal":    range(10, 20),
    "libero_object":  range(20, 30),
    "libero_spatial": range(30, 40),
}

def episode_belongs_to_suite(task_idx: int, suite):
    if suite is None:
        return True
    if suite not in SUITE_TASK_RANGES:
        raise ValueError(f"Unknown suite {suite!r}. "
                         f"Known: {list(SUITE_TASK_RANGES)}")
    return task_idx in SUITE_TASK_RANGES[suite]

target_episodes = [
    ep_idx for ep_idx, (ti, _) in ep_to_task.items()
    if episode_belongs_to_suite(ti, target_suite)
]
print(f"Episodes to process: {len(target_episodes)}")

Built ep_to_task with 1693 entries
Episodes to process: 428


In [ ]:
print("Probing max sequence length over unique instructions...")
unique_tasks = list(set(ep_to_task[ep][1] for ep in target_episodes))

dummy_main  = np.zeros((256, 256, 3), dtype=np.float32)
dummy_wrist = np.zeros((256, 256, 3), dtype=np.float32)

max_seq_len = 0
for task in tqdm(unique_tasks):
    _, mask = planner.extract_features_batch(
        [dummy_main], [dummy_wrist], [task]
    )
    max_seq_len = max(max_seq_len, mask.shape[1])
print(f"max_seq_len = {max_seq_len}")


Probing max sequence length over unique instructions...


  0%|          | 0/10 [00:00<?, ?it/s]

max_seq_len = 141


In [ ]:
if Path(out_h5_path).exists():
    Path(out_h5_path).unlink()

f = h5py.File(out_h5_path, "w")
data_grp = f.create_group("data")

f.attrs["suite"] = target_suite or "all"
f.attrs["source_dataset"] = in_dataset_repo
f.attrs["model_name"] = "Qwen/Qwen3-VL-4B-Instruct"
f.attrs["vlm_hidden_dim"] = vlm_dim
f.attrs["num_layers"] = len(layer_indices)
f.attrs["layer_indices"] = layer_indices
f.attrs["main_camera_key"] = main_camera_key
f.attrs["wrist_camera_key"] = wrist_camera_key
f.attrs["vlm_stride"] = vlm_stride
f.attrs["max_seq_len"] = max_seq_len


In [ ]:
def fetch_episode_frames(dataset, ep_idx, main_key, wrist_key):
    ep_meta = dataset.meta.episodes[ep_idx]
    if isinstance(ep_meta, dict):
        from_idx = ep_meta.get("from_index", ep_meta.get("dataset_from_index"))
        to_idx = ep_meta.get("to_index",   ep_meta.get("dataset_to_index"))
    else:
        from_idx = getattr(ep_meta, "from_index", getattr(ep_meta, "dataset_from_index", None))
        to_idx = getattr(ep_meta, "to_index", getattr(ep_meta, "dataset_to_index", None))

    imgs_main, imgs_wrist, states, actions = [], [], [], []
    instr = None
    for gi in range(from_idx, to_idx):
        sample = dataset[gi]
        imgs_main.append(to_hwc_uint8_float(sample[main_key]))
        imgs_wrist.append(to_hwc_uint8_float(sample[wrist_key]))
        states.append(sample["observation.state"].cpu().numpy()
                      if isinstance(sample["observation.state"], torch.Tensor)
                      else np.asarray(sample["observation.state"]))
        actions.append(sample["action"].cpu().numpy()
                       if isinstance(sample["action"], torch.Tensor)
                       else np.asarray(sample["action"]))
        if instr is None:
            t = sample.get("task", None)
            if isinstance(t, str):
                instr = t
            else:
                ti = sample.get("task_index", None)
                if ti is None:
                    ti = (ep_meta.get("task_index", 0)
                          if isinstance(ep_meta, dict)
                          else getattr(ep_meta, "task_index", 0))
                instr = dataset.meta.tasks[int(ti)]

    return (
        np.stack(imgs_main).astype(np.float32),
        np.stack(imgs_wrist).astype(np.float32),
        np.stack(states).astype(np.float32),
        np.stack(actions).astype(np.float32),
        instr,
    )


for ep_idx in tqdm(target_episodes, desc="Extracting demos"):
    imgs_m, imgs_w, states, actions, instruction = fetch_episode_frames(
        dataset, ep_idx, main_camera_key, wrist_camera_key
    )
    T = imgs_m.shape[0]

    slow_src, per_frame_slow = compute_vlm_indices(T, vlm_stride)
    T_slow = len(slow_src)
    imgs_m_slow = imgs_m[slow_src]
    imgs_w_slow = imgs_w[slow_src]

    hidden, mask = extract_slow_stream(
        planner, imgs_m_slow, imgs_w_slow, instruction,
        batch_size=batch_size, layer_indices=layer_indices,
    )

    mask_np = mask.cpu().numpy()
    hidden_uint16 = bf16_to_uint16(hidden)

    # Pad / truncate to max_seq_len
    pad_len = max_seq_len - mask_np.shape[1]
    if pad_len > 0:
        mask_np       = np.pad(mask_np, ((0, 0), (0, pad_len)))
        hidden_uint16 = np.pad(hidden_uint16,
                               ((0, 0), (0, 0), (0, pad_len), (0, 0)))
    elif pad_len < 0:
        mask_np       = mask_np[:, :max_seq_len]
        hidden_uint16 = hidden_uint16[:, :, :max_seq_len, :]

    ep_grp = data_grp.create_group(f"demo_{ep_idx}")
    ep_grp.attrs["instruction"] = instruction
    ti, tname = ep_to_task[ep_idx]
    ep_grp.attrs["task_index"] = ti
    ep_grp.attrs["task_name"] = tname
    ep_grp.attrs["num_frames"] = T
    ep_grp.attrs["num_slow_frames"] = T_slow

    obs_grp = ep_grp.create_group("obs")
    obs_grp.create_dataset("state", data=states,
                           chunks=(1, states.shape[1]), compression="lzf")
    obs_grp.create_dataset(
        "img_features",
        data=hidden_uint16,
        chunks=(1,) + hidden_uint16.shape[1:],
        compression="lzf",
    )
    ep_grp.create_dataset("action", data=actions,
                          chunks=(1, actions.shape[1]), compression="lzf")
    ep_grp.create_dataset("vlm_source_idx", data=slow_src,
                          chunks=slow_src.shape, compression="lzf")
    ep_grp.create_dataset("vlm_frame_idx", data=per_frame_slow,
                          chunks=per_frame_slow.shape, compression="lzf")
    ep_grp.create_dataset("text_mask", data=mask_np,
                          chunks=(1,) + mask_np.shape[1:], compression="lzf")

f.close()
print(f"Wrote {out_h5_path}")


Extracting demos:   0%|          | 0/428 [00:00<?, ?it/s]

Wrote /content/libero_features_v2.h5


In [ ]:
print(type(dataset.meta.tasks))
print(dataset.meta.tasks)

<class 'pandas.core.frame.DataFrame'>
                                                    task_index
task                                                          
put the white mug on the left plate and put the...           0
put the white mug on the plate and put the choc...           1
put the yellow and white mug in the microwave a...           2
turn on the stove and put the moka pot on it                 3
put both the alphabet soup and the cream cheese...           4
put both the alphabet soup and the tomato sauce...           5
put both moka pots on the stove                              6
put both the cream cheese box and the butter in...           7
put the black bowl in the bottom drawer of the ...           8
pick up the book and place it in the back compa...           9
put the bowl on the plate                                   10
put the wine bottle on the rack                             11
open the top drawer and put the bowl inside                 12
put the cream che

In [ ]:
from collections import Counter

# task_index -> instruction string
idx_to_task = {int(row["task_index"]): str(idx)
               for idx, row in dataset.meta.tasks.iterrows()}

task_counts = Counter(ep_to_task[ep][0] for ep in target_episodes)
print("Demos per task_index in libero_goal:")
for ti in sorted(task_counts):
    print(f"  [{ti}] {idx_to_task[ti]}: {task_counts[ti]}")

Demos per task_index in libero_goal:
  [10] put the bowl on the plate: 49
  [11] put the wine bottle on the rack: 36
  [12] open the top drawer and put the bowl inside: 36
  [13] put the cream cheese in the bowl: 40
  [14] put the wine bottle on top of the cabinet: 47
  [15] push the plate to the front of the stove: 33
  [16] turn on the stove: 50
  [17] put the bowl on the stove: 48
  [18] put the bowl on top of the cabinet: 46
  [19] open the middle drawer of the cabinet: 43


In [ ]:
sample_instructions = [ep_to_task[ep][1] for ep in target_episodes[:5]]
print("Sample instructions from ep_to_task:")
for instr in sample_instructions:
    print(f"  {instr!r}")

Sample instructions from ep_to_task:
  'put the bowl on the plate'
  'put the wine bottle on the rack'
  'put the wine bottle on the rack'
  'open the top drawer and put the bowl inside'
  'open the top drawer and put the bowl inside'


In [ ]:
for ep_idx in [0, 100, 200, 300, 400]:
    ti, name = ep_to_task[ep_idx]
    print(f"  ep {ep_idx}: ti={ti}, name={name!r}")

  ep 0: ti=0, name='put the white mug on the left plate and put the yellow and white mug on the right plate'
  ep 100: ti=6, name='put both moka pots on the stove'
  ep 200: ti=5, name='put both the alphabet soup and the tomato sauce in the basket'
  ep 300: ti=8, name='put the black bowl in the bottom drawer of the cabinet and close it'
  ep 400: ti=15, name='push the plate to the front of the stove'


In [ ]:
ep_meta = dataset.meta.episodes[0]
print(f"Type of ep_meta: {type(ep_meta)}")
print(f"Repr: {repr(ep_meta)[:300]}")
if isinstance(ep_meta, dict):
    print(f"Keys: {list(ep_meta.keys())}")

Type of ep_meta: <class 'dict'>
Repr: {'episode_index': 0, 'length': 214, 'dataset_from_index': 0, 'dataset_to_index': 214, 'data/chunk_index': 0, 'data/file_index': 0, 'videos/observation.images.image/chunk_index': 0, 'videos/observation.images.image/file_index': 0, 'videos/observation.images.image/from_timestamp': 0.0, 'videos/observa
Keys: ['episode_index', 'length', 'dataset_from_index', 'dataset_to_index', 'data/chunk_index', 'data/file_index', 'videos/observation.images.image/chunk_index', 'videos/observation.images.image/file_index', 'videos/observation.images.image/from_timestamp', 'videos/observation.images.image/to_timestamp', 'videos/observation.images.image2/chunk_index', 'videos/observation.images.image2/file_index', 'videos/observation.images.image2/from_timestamp', 'videos/observation.images.image2/to_timestamp']


In [ ]:
import h5py
import numpy as np
import torch

with h5py.File(out_h5_path, "r") as f:
    print("File attributes")
    for k, v in f.attrs.items():
        print(f"  {k}: {v}")
    print()

    demos = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
    print(f"Demo count")
    print(f"  Demos in file: {len(demos)}")
    print(f"  Expected     : {len(target_episodes)}")
    assert len(demos) == len(target_episodes), "Demo count mismatch!"
    print()

    print("Per-demo structure (first 3)")
    for d in demos[:3]:
        g = f["data"][d]
        print(f"  {d}:")
        print(f"    instruction: {g.attrs['instruction']!r}")
        print(f"    task_index: {g.attrs['task_index']}")
        print(f"    task_name: {g.attrs['task_name']!r}")
        print(f"    num_frames: {g.attrs['num_frames']}")
        print(f"    num_slow_frames: {g.attrs['num_slow_frames']}")
        print(f"    obs/state: {g['obs/state'].shape} {g['obs/state'].dtype}")
        print(f"    obs/img_features: {g['obs/img_features'].shape} {g['obs/img_features'].dtype}")
        print(f"    action: {g['action'].shape} {g['action'].dtype}")
        print(f"    vlm_source_idx: {g['vlm_source_idx'].shape}")
        print(f"    vlm_frame_idx: {g['vlm_frame_idx'].shape}")
        print(f"    text_mask: {g['text_mask'].shape} {g['text_mask'].dtype}")

    print()
    print("Shape consistency")
    expected_layers = f.attrs["num_layers"]
    expected_seq_len = f.attrs["max_seq_len"]
    expected_vlm_dim = f.attrs["vlm_hidden_dim"]
    expected_stride = f.attrs["vlm_stride"]

    bad = []
    for d in demos:
        g = f["data"][d]
        T = g.attrs["num_frames"]
        T_slow_expected = (T + expected_stride - 1) // expected_stride
        T_slow_actual = g.attrs["num_slow_frames"]

        if T_slow_actual != T_slow_expected:
            bad.append((d, f"T_slow expected {T_slow_expected}, got {T_slow_actual}"))

        feat_shape = g["obs/img_features"].shape
        if feat_shape != (T_slow_actual, expected_layers, expected_seq_len, expected_vlm_dim):
            bad.append((d, f"img_features shape {feat_shape} unexpected"))

        if g["action"].shape[0] != T:
            bad.append((d, f"action len {g['action'].shape[0]} != T={T}"))
        if g["obs/state"].shape[0] != T:
            bad.append((d, f"state len {g['obs/state'].shape[0]} != T={T}"))

        vfi = g["vlm_frame_idx"][:]
        if vfi.min() < 0 or vfi.max() >= T_slow_actual:
            bad.append((d, f"vlm_frame_idx out of range: [{vfi.min()}, {vfi.max()}]"))

    if bad:
        print(f"  FAIL: {len(bad)} demos with issues:")
        for d, msg in bad[:10]:
            print(f"    {d}: {msg}")
    else:
        print(f"  PASS: all {len(demos)} demos have consistent shapes")
    print()

    print("Feature content check")
    g = f["data"][demos[0]]
    # uint16 storage
    feat_u16 = g["obs/img_features"][:5]  # first 5 slow frames
    feat_bf16 = torch.from_numpy(feat_u16.view(np.int16)).view(torch.bfloat16)
    feat_f32 = feat_bf16.float()

    print(f"  Sampled tensor: {tuple(feat_f32.shape)}")
    print(f"  mean: {feat_f32.mean().item():.6f}")
    print(f"  std: {feat_f32.std().item():.6f}")
    print(f"  min: {feat_f32.min().item():.6f}")
    print(f"  max: {feat_f32.max().item():.6f}")
    print(f"  %zero: {(feat_f32 == 0).float().mean().item() * 100:.2f}%")
    print(f"  %nan: {feat_f32.isnan().float().mean().item() * 100:.2f}%")
    print(f"  %inf: {feat_f32.isinf().float().mean().item() * 100:.2f}%")

    print()

    print("Text mask check")
    mask = g["text_mask"][:5]  # (5, L)
    valid_per_row = mask.sum(axis=1)
    print(f"  Valid tokens per slow frame (first 5): {valid_per_row.tolist()}")
    print(f"  Max seq len: {mask.shape[1]}")
    print(f"  Per-frame valid avg: {valid_per_row.mean():.1f}")

    if not np.all(valid_per_row == valid_per_row[0]):
        print(f"  WARNING: varying valid-token counts within one demo, "
              f"that's suspicious unless image resolution varies frame to frame")
    print()

    pad_position = mask.shape[1] - 1
    if not mask[0, pad_position]:
        feat_at_pad = feat_f32[0, :, pad_position, :]
        print(f"  Padded-position feature norm: {feat_at_pad.norm().item():.6f} "
              f"(should be ~0)")
    print()

    print("Instruction consistency")
    mismatches = 0
    for d in demos:
        ep_idx = int(d.split("_")[1])
        attr_instr   = f["data"][d].attrs["instruction"]
        if isinstance(attr_instr, bytes):
            attr_instr = attr_instr.decode()
        expected_ti, expected_name = ep_to_task[ep_idx]
        attr_ti = int(f["data"][d].attrs["task_index"])
        if attr_instr.strip() != expected_name.strip():
            mismatches += 1
            if mismatches <= 3:
                print(f"  {d}: stored={attr_instr!r}, expected={expected_name!r}")
        if attr_ti != expected_ti:
            mismatches += 1
            if mismatches <= 3:
                print(f"  {d}: stored ti={attr_ti}, expected ti={expected_ti}")
    if mismatches == 0:
        print(f"  PASS: all {len(demos)} demos have consistent instructions and task_index")
    else:
        print(f"  FAIL: {mismatches} mismatches")
    print()

    print("Storage summary")
    import os
    file_size_gb = os.path.getsize(out_h5_path) / 1e9
    total_slow_frames = sum(f["data"][d].attrs["num_slow_frames"] for d in demos)
    total_frames = sum(f["data"][d].attrs["num_frames"] for d in demos)
    bytes_per_slow_frame = (
        expected_layers * expected_seq_len * expected_vlm_dim * 2  # bf16 stored as uint16
    )
    print(f"  File size: {file_size_gb:.2f} GB")
    print(f"  Total frames: {total_frames}")
    print(f"  Total slow frames: {total_slow_frames}")
    print(f"  Bytes per slow frame: {bytes_per_slow_frame:,} "
          f"({bytes_per_slow_frame / 1e6:.2f} MB)")
    print(f"  Expected raw feat size: "
          f"{total_slow_frames * bytes_per_slow_frame / 1e9:.2f} GB "
          f"(lzf compression brings it down)")

print()
print("Probe complete.")

=== File attributes ===
  layer_indices: [12 18 24 30]
  main_camera_key: observation.images.image
  max_seq_len: 141
  model_name: Qwen/Qwen3-VL-4B-Instruct
  num_layers: 4
  source_dataset: lerobot/libero
  suite: libero_goal
  vlm_hidden_dim: 2560
  vlm_stride: 8
  wrist_camera_key: observation.images.image2

=== Demo count ===
  Demos in file: 428
  Expected     : 428

=== Per-demo structure (first 3) ===
  demo_379:
    instruction      : 'put the bowl on the plate'
    task_index       : 10
    task_name        : 'put the bowl on the plate'
    num_frames       : 112
    num_slow_frames  : 14
    obs/state        : (112, 8) float32
    obs/img_features : (14, 4, 141, 2560) uint16
    action           : (112, 7) float32
    vlm_source_idx   : (14,)
    vlm_frame_idx    : (112,)
    text_mask        : (14, 141) bool
  demo_380:
    instruction      : 'put the wine bottle on the rack'
    task_index       : 11
    task_name        : 'put the wine bottle on the rack'
    num_frames  

In [ ]:
import h5py, numpy as np, torch

with h5py.File(out_h5_path, "r") as f:
    demos = sorted(f["data"].keys(), key=lambda x: int(x.split("_")[1]))
    # Sample from 3 different demos to be safe
    feats = []
    for d in demos[:3]:
        x = f["data"][d]["obs/img_features"][:5]   # 5 slow frames each
        feats.append(torch.from_numpy(x.view(np.int16)).view(torch.bfloat16).float())
    feat = torch.cat(feats, dim=0)   # (15, 4, 141, 2560)?
    masks = []
    for d in demos[:3]:
        masks.append(torch.from_numpy(f["data"][d]["text_mask"][:5]).bool())
    mask = torch.cat(masks, dim=0)   # (15, 141)

# Only look at valid (non-padded) positions
valid = mask.unsqueeze(1).unsqueeze(-1).expand_as(feat)   # (15, 4, 141, 2560)
print("Per-layer stats (valid positions only):")
print(f"  Layer idx (file order): {list(range(4))} => actual VLM layers [12, 18, 24, 30]")
for layer_i in range(4):
    f_l = feat[:, layer_i][valid[:, layer_i]]
    print(f"  Layer {layer_i} (VLM {[12,18,24,30][layer_i]}): "
          f"mean={f_l.mean().item():+.4f}, "
          f"std={f_l.std().item():.4f}, "
          f"min={f_l.min().item():+.2f}, "
          f"max={f_l.max().item():+.2f}")

# Find the % of values > 100 in magnitude (outliers)
outliers = (feat.abs() > 100).float().mean().item()
print(f"\n  % values with |x| > 100: {outliers * 100:.4f}%")
print(f"  % values with |x| > 500: {(feat.abs() > 500).float().mean().item() * 100:.4f}%")

Per-layer stats (valid positions only):
  Layer idx (file order): [0, 1, 2, 3] => actual VLM layers [12, 18, 24, 30]
  Layer 0 (VLM 12): mean=+0.0337, std=13.2726, min=-2096.00, max=+7712.00
  Layer 1 (VLM 18): mean=+0.0497, std=15.9081, min=-2096.00, max=+7936.00
  Layer 2 (VLM 24): mean=+0.0646, std=15.9524, min=-2096.00, max=+7936.00
  Layer 3 (VLM 30): mean=+0.0617, std=16.1924, min=-2096.00, max=+7936.00

  % values with |x| > 100: 0.0027%
  % values with |x| > 500: 0.0010%


In [ ]:
api = HfApi()
api.create_repo(repo_id=out_dataset_id, repo_type="dataset", exist_ok=True)
api.upload_file(
    path_or_fileobj=out_h5_path,
    path_in_repo="libero_features_v2.h5",
    repo_id=out_dataset_id,
    repo_type="dataset",
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ent/libero_features_v2.h5:   0%|          |  525kB / 19.0GB            

CommitInfo(commit_url='https://huggingface.co/datasets/lnxdre4d/libero_features_v2/commit/1f46d78a178224b8b8baa2c14ce1f14cac4990d6', commit_message='Upload libero_features_v2.h5 with huggingface_hub', commit_description='', oid='1f46d78a178224b8b8baa2c14ce1f14cac4990d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/lnxdre4d/libero_features_v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='lnxdre4d/libero_features_v2'), pr_revision=None, pr_num=None)

In [ ]:
print("Upload complete. Disconnecting runtime to save compute units...")
runtime.unassign()

Upload complete. Disconnecting runtime to save compute units...
